In [2]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import pandas as pd

air_korea_final = pd.read_pickle("/Users/drewbaldwin/PM2_5 Research/air_korea_final_imputed.pkl")
air_korea_final.head()


,Datetime,SO2,CO,O3,NO2,PM10,PM25,Station_ID,Year,lon,...,major_roads_count_3km,total_road_length_3km,urban_landuse_area_m2_3km,green_space_area_3km,building_footprint_area_3km,railway_length_3km,dist_to_coast_km,dist_to_major_road_km,industrial_area_m2_3km,traffic_points_count_3km
0,2016-01-01 00:00:00,7.0,1000.0,2.0,76.0,77.0,53.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0
1,2016-01-01 01:00:00,7.0,1100.0,2.0,77.0,70.0,48.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0
2,2016-01-01 02:00:00,7.0,1200.0,2.0,78.0,75.0,53.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0
3,2016-01-01 03:00:00,6.0,1400.0,2.0,78.0,77.0,53.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0
4,2016-01-01 04:00:00,6.0,1500.0,2.0,77.0,83.0,52.0,111121,2016,126.9747,...,613.0,923909.036845,2.952433e+06,9.417996e+06,4.942823e+06,114737.280122,10.830496,0.102038,1489.332477,159.0


In [4]:


# ---------------- Config ----------------
IN_PATH = "/Users/drewbaldwin/PM2_5 Research/air_korea_final_imputed.pkl"

DISTANCE_THRESHOLD_KM = 25.0   # candidate-edge cutoff -- tune using the diagnostic below,
                                # not fixed; re-run everything downstream if this changes
CALM_WIND_SPEED = 1.0          # below this, direction is unreliable -- fall back to distance-only

df = pd.read_pickle(IN_PATH)

# ---------------- Station lookup + distance/bearing matrices ----------------
stations = (
    df[["Station_ID", "lat", "lon"]]
    .drop_duplicates("Station_ID")
    .sort_values("Station_ID")
    .reset_index(drop=True)
)
station_order = stations["Station_ID"].tolist()
n_stations = len(station_order)

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    """Compass bearing (0-360, 0=N) from point 1 to point 2."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

lats = stations["lat"].values
lons = stations["lon"].values
distance_matrix = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
bearing_matrix = bearing_deg(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
np.fill_diagonal(distance_matrix, np.inf)  # no self-edges

# ---------------- Diagnostic: does spatial correlation actually decay around 25km? ----------------
# Estimated over the full multi-year record (stable), not a single day -- use this to
# pick DISTANCE_THRESHOLD_KM instead of assuming it.
pm25_wide = df.pivot(index="Datetime", columns="Station_ID", values="PM25")[station_order]
corr_matrix = pm25_wide.corr().values

iu = np.triu_indices(n_stations, k=1)
pair_distances = distance_matrix[iu]
pair_corrs = corr_matrix[iu]

dist_bins = np.arange(0, 205, 10)
bin_idx = np.digitize(pair_distances, dist_bins)
binned = pd.DataFrame({"dist": pair_distances, "corr": pair_corrs, "bin": bin_idx}).groupby("bin").mean()
print("Distance (km) -> mean pairwise PM2.5 correlation (full history):")
print(binned.to_string(index=False))

# ---------------- Candidate edges (fixed structure, not rebuilt per day) ----------------
within_threshold = distance_matrix <= DISTANCE_THRESHOLD_KM
edge_i, edge_j = np.where(within_threshold)

candidate_edges = pd.DataFrame({
    "Station_i": [station_order[i] for i in edge_i],
    "Station_j": [station_order[j] for j in edge_j],
    "distance_km": distance_matrix[edge_i, edge_j],
    "bearing_ij": bearing_matrix[edge_i, edge_j],  # direction from i to j
})
print(f"\n{len(candidate_edges)} candidate directed edges within {DISTANCE_THRESHOLD_KM} km "
      f"({len(candidate_edges) // 2} station pairs)")

# ---------------- Daily wind vector per station ----------------
# winddirection_10m is the direction wind blows FROM (met convention). Convert to U/V
# (blowing-TOWARD components) before averaging -- never average raw degrees, that
# breaks at the 0/360 wraparound.
wind = df[["Datetime", "Station_ID", "windspeed_10m", "winddirection_10m"]].copy()
wind["Date"] = wind["Datetime"].dt.date
wind_rad = np.radians(270 - wind["winddirection_10m"])
wind["U"] = -wind["windspeed_10m"] * np.cos(wind_rad)
wind["V"] = -wind["windspeed_10m"] * np.sin(wind_rad)

daily_wind = wind.groupby(["Station_ID", "Date"])[["U", "V"]].mean().reset_index()
daily_wind["wind_speed"] = np.hypot(daily_wind["U"], daily_wind["V"])
daily_wind["wind_bearing"] = (np.degrees(np.arctan2(daily_wind["U"], daily_wind["V"])) + 360) % 360
print(f"\n{daily_wind.shape[0]:,} station-days of daily wind computed")

# ---------------- Daily directed edge weights ----------------
def circular_diff(a, b):
    d = np.abs(a - b) % 360
    return np.minimum(d, 360 - d)

wind_i = daily_wind.rename(columns={
    "Station_ID": "Station_i", "wind_speed": "wind_speed_i", "wind_bearing": "wind_bearing_i",
})[["Station_i", "Date", "wind_speed_i", "wind_bearing_i"]]

daily_networks = candidate_edges.merge(wind_i, on="Station_i", how="left")

angle_diff = circular_diff(daily_networks["wind_bearing_i"], daily_networks["bearing_ij"])
alignment = np.clip(np.cos(np.radians(angle_diff)), 0, None)  # 0 beyond 90 degrees off
is_calm = daily_networks["wind_speed_i"] < CALM_WIND_SPEED
alignment = np.where(is_calm, 1.0, alignment)  # no reliable direction -> distance-only

distance_decay = 1 - (daily_networks["distance_km"] / DISTANCE_THRESHOLD_KM)  # linear falloff
daily_networks["weight"] = distance_decay * alignment
daily_networks = daily_networks[daily_networks["weight"] > 0].reset_index(drop=True)

print(f"\n{len(daily_networks):,} directed day-edges with positive weight "
      f"across {daily_networks['Date'].nunique():,} days")
daily_networks.head()

daily_networks.to_pickle("/Users/drewbaldwin/PM2_5 Research/daily_wind_networks.pkl")


Distance (km) -> mean pairwise PM2.5 correlation (full history):
      dist     corr
  6.608860 0.884959
 15.036805 0.854720
 24.988101 0.821834
 34.794171 0.798287
 44.615075 0.773242
 55.024939 0.758356
 64.661785 0.737926
 75.361888 0.714447
 85.335618 0.699449
 94.522432 0.691511
105.053511 0.683306
114.935195 0.664480
124.642793 0.648346
135.169240 0.651627
144.633170 0.641944
154.876302 0.621718
165.270861 0.619992
175.153685 0.615954
185.145297 0.596862
195.144220 0.579030
283.061944 0.506502

2646 candidate directed edges within 25.0 km (1323 station pairs)

385,792 station-days of daily wind computed

2,971,102 directed day-edges with positive weight across 2,192 days


In [5]:
# Remove the shared national-scale signal before looking at distance decay --
# raw correlation above is dominated by everyone-moves-together confounding
# (synoptic weather, transboundary transport, seasonal heating), not genuine
# local spatial dependence.
national_mean = pm25_wide.mean(axis=1)
pm25_residual = pm25_wide.sub(national_mean, axis=0)

resid_corr_matrix = pm25_residual.corr().values
pair_resid_corrs = resid_corr_matrix[iu]

binned_resid = pd.DataFrame(
    {"dist": pair_distances, "corr": pair_resid_corrs, "bin": bin_idx}
).groupby("bin").mean()
print("Distance (km) -> mean pairwise PM2.5 RESIDUAL correlation (national signal removed):")
print(binned_resid.to_string(index=False))


Distance (km) -> mean pairwise PM2.5 RESIDUAL correlation (national signal removed):
      dist      corr
  6.608860  0.655855
 15.036805  0.546029
 24.988101  0.443197
 34.794171  0.373306
 44.615075  0.349192
 55.024939  0.328364
 64.661785  0.279253
 75.361888  0.224258
 85.335618  0.181912
 94.522432  0.172876
105.053511  0.101541
114.935195  0.057746
124.642793  0.057215
135.169240  0.031025
144.633170  0.023492
154.876302  0.038699
165.270861 -0.006881
175.153685 -0.047759
185.145297 -0.084639
195.144220 -0.093362
283.061944 -0.259107


In [6]:
import numpy as np
from scipy.optimize import curve_fit

# ---------------- Fit a decay curve to the deseasonalized correlation-distance data ----------------
# Fit on the raw pairwise points (not the binned means) for a proper nonlinear least-squares
# fit, restricted to distances below the observed zero-crossing (~160km) -- beyond that,
# correlation goes negative (mechanical demeaning bias + likely a small-sample Jeju effect),
# which a monotonic decay-to-zero model can't represent and shouldn't be fit against.
FIT_MAX_DISTANCE_KM = 160

fit_mask = pair_distances <= FIT_MAX_DISTANCE_KM
fit_dist = pair_distances[fit_mask]
fit_corr = pair_resid_corrs[fit_mask]
print(f"Fitting on {fit_mask.sum():,} of {len(pair_distances):,} station pairs (<= {FIT_MAX_DISTANCE_KM} km)")

def exponential_decay(d, sill, range_km):
    return sill * np.exp(-d / range_km)

def gaussian_decay(d, sill, range_km):
    return sill * np.exp(-(d / range_km) ** 2)

def r_squared(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

results = {}
for name, func in [("exponential", exponential_decay), ("gaussian", gaussian_decay)]:
    popt, _ = curve_fit(func, fit_dist, fit_corr, p0=[0.6, 50], maxfev=10000)
    pred = func(fit_dist, *popt)
    sill, rng = popt
    results[name] = {"sill": sill, "range_km": rng, "r2": r_squared(fit_corr, pred)}
    print(f"{name:12s}: sill={sill:.3f}  range={rng:.1f} km  R^2={results[name]['r2']:.4f}")

best_model = max(results, key=lambda k: results[k]["r2"])
print(f"\nBest-fitting model: {best_model}, range = {results[best_model]['range_km']:.1f} km "
      "(standard geostatistics convention: distance at which correlation decays to sill/e, ~37% of peak)")

# Distances at which the fitted curve crosses a few practical correlation thresholds
print()
for name, func in [("exponential", exponential_decay), ("gaussian", gaussian_decay)]:
    sill, rng = results[name]["sill"], results[name]["range_km"]
    for target in [0.5, 0.25, 0.1, 0.05]:
        if target < sill:
            d_target = -rng * np.log(target / sill) if name == "exponential" else rng * np.sqrt(-np.log(target / sill))
            print(f"{name:12s}: correlation = {target:.2f} at {d_target:6.1f} km")


Fitting on 6,942 of 15,400 station pairs (<= 160 km)
exponential : sill=0.717  range=57.6 km  R^2=0.6071
gaussian    : sill=0.535  range=79.2 km  R^2=0.5891

Best-fitting model: exponential, range = 57.6 km (standard geostatistics convention: distance at which correlation decays to sill/e, ~37% of peak)

exponential : correlation = 0.50 at   20.7 km
exponential : correlation = 0.25 at   60.7 km
exponential : correlation = 0.10 at  113.4 km
exponential : correlation = 0.05 at  153.4 km
gaussian    : correlation = 0.50 at   20.7 km
gaussian    : correlation = 0.25 at   69.1 km
gaussian    : correlation = 0.10 at  102.5 km
gaussian    : correlation = 0.05 at  121.9 km


In [12]:
import numpy as np
import pandas as pd

# ---------------- Sanity check the bearing conversion before trusting anything downstream ----------------
# winddirection_10m = direction wind blows FROM (met convention). Blowing-TOWARD bearing = +180.
test_from = np.array([0, 90, 180, 270])
test_toward = (test_from + 180) % 360
assert list(test_toward) == [180, 270, 0, 90], "wind bearing flip is wrong"
print("Wind bearing sanity check: from", dict(zip(test_from, test_toward)), "(should blow toward the opposite compass point)")

# ---------------- Corrected daily wind vector per station (for network building) ----------------
wind = df[["Datetime", "Station_ID", "windspeed_10m", "winddirection_10m"]].copy()
wind["Date"] = wind["Datetime"].dt.date

wind_bearing_toward = (wind["winddirection_10m"] + 180) % 360
toward_rad = np.radians(wind_bearing_toward)
wind["U"] = wind["windspeed_10m"] * np.sin(toward_rad)  # eastward component of where it's blowing TOWARD
wind["V"] = wind["windspeed_10m"] * np.cos(toward_rad)  # northward component

daily_wind = wind.groupby(["Station_ID", "Date"])[["U", "V"]].mean().reset_index()
daily_wind["wind_speed"] = np.hypot(daily_wind["U"], daily_wind["V"])
daily_wind["wind_bearing"] = (np.degrees(np.arctan2(daily_wind["U"], daily_wind["V"])) + 360) % 360
print(f"\n{daily_wind.shape[0]:,} station-days of daily wind recomputed with corrected bearing")

# ---------------- Corrected hourly wind (wide matrix, for the lag diagnostics) ----------------
wind_hourly = df[["Datetime", "Station_ID", "windspeed_10m", "winddirection_10m"]].copy()
wind_bearing_toward_h = (wind_hourly["winddirection_10m"] + 180) % 360
toward_rad_h = np.radians(wind_bearing_toward_h)
wind_hourly["U"] = wind_hourly["windspeed_10m"] * np.sin(toward_rad_h)
wind_hourly["V"] = wind_hourly["windspeed_10m"] * np.cos(toward_rad_h)

U_wide = wind_hourly.pivot(index="Datetime", columns="Station_ID", values="U")[station_order].to_numpy()
V_wide = wind_hourly.pivot(index="Datetime", columns="Station_ID", values="V")[station_order].to_numpy()
wind_bearing_wide = (np.degrees(np.arctan2(U_wide, V_wide)) + 360) % 360  # hours x stations

def circular_diff(a, b):
    d = np.abs(a - b) % 360
    return np.minimum(d, 360 - d)

# ---------------- Rerun lag-1 test with corrected bearings ----------------
from collections import defaultdict

LAG = 1
MIN_WIND_SPEED = 2.0
DIST_BINS = [0, 15, 30, 45, 60]
DIST_BIN_LABELS = [f"({DIST_BINS[k]}, {DIST_BINS[k+1]}]" for k in range(len(DIST_BINS) - 1)]
ALIGN_EDGES = [-1.01, -0.5, 0.5, 1.01]
ALIGN_LABELS = ["opposed", "neutral", "aligned"]

stats = defaultdict(lambda: {"n": 0, "sx": 0.0, "sy": 0.0, "sxx": 0.0, "syy": 0.0, "sxy": 0.0})

for _, row in candidate_edges.iterrows():
    i, j = station_index[row["Station_i"]], station_index[row["Station_j"]]
    dist, bearing_ij = row["distance_km"], row["bearing_ij"]

    d_bin = np.digitize([dist], DIST_BINS)[0] - 1
    if not (0 <= d_bin < len(DIST_BIN_LABELS)):
        continue
    d_label = DIST_BIN_LABELS[d_bin]

    speed_i = np.hypot(U_wide[:-LAG, i], V_wide[:-LAG, i])
    alignment_h = np.cos(np.radians(circular_diff(wind_bearing_wide[:-LAG, i], bearing_ij)))
    x, y = resid_values[:-LAG, i], resid_values[LAG:, j]

    valid = ~np.isnan(x) & ~np.isnan(y) & ~np.isnan(alignment_h) & (speed_i >= MIN_WIND_SPEED)
    x, y, alignment_h = x[valid], y[valid], alignment_h[valid]
    align_idx = np.digitize(alignment_h, ALIGN_EDGES) - 1

    for a_idx, a_label in enumerate(ALIGN_LABELS):
        mask = align_idx == a_idx
        if not mask.any():
            continue
        xs, ys = x[mask], y[mask]
        s = stats[(d_label, a_label)]
        s["n"] += len(xs); s["sx"] += xs.sum(); s["sy"] += ys.sum()
        s["sxx"] += (xs ** 2).sum(); s["syy"] += (ys ** 2).sum(); s["sxy"] += (xs * ys).sum()

rows = []
for (d_label, a_label), s in stats.items():
    n = s["n"]
    if n < 100:
        continue
    mx, my = s["sx"] / n, s["sy"] / n
    cov = s["sxy"] / n - mx * my
    corr = cov / np.sqrt((s["sxx"] / n - mx**2) * (s["syy"] / n - my**2))
    rows.append({"dist_bin": d_label, "align_bin": a_label, "n": n, "corr": corr})

result = pd.DataFrame(rows)
print(f"\nCORRECTED lag={LAG} test: resid_i(h) vs resid_j(h+1)\n")
print(result.pivot(index="dist_bin", columns="align_bin", values="corr")[ALIGN_LABELS].to_string())


Wind bearing sanity check: from {np.int64(0): np.int64(180), np.int64(90): np.int64(270), np.int64(180): np.int64(0), np.int64(270): np.int64(90)} (should blow toward the opposite compass point)

385,792 station-days of daily wind recomputed with corrected bearing

CORRECTED lag=1 test: resid_i(h) vs resid_j(h+1)

align_bin   opposed   neutral   aligned
dist_bin                               
(0, 15]    0.578399  0.589012  0.608108
(15, 30]   0.448148  0.466171  0.488721


In [16]:
import numpy as np
import pandas as pd

# ---------------- Config ----------------
DISTANCE_THRESHOLD_KM = 57.6   # candidate-edge cutoff = the fitted exponential decay range
WIND_WEIGHT = 0.15             # modest modulation, matching the measured relative effect sizes

# ---------------- Station lookup + distance/bearing matrices ----------------
stations = (
    df[["Station_ID", "lat", "lon"]]
    .drop_duplicates("Station_ID")
    .sort_values("Station_ID")
    .reset_index(drop=True)
)
station_order = stations["Station_ID"].tolist()
station_index = {sid: i for i, sid in enumerate(station_order)}
n_stations = len(station_order)

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

lats = stations["lat"].values
lons = stations["lon"].values
distance_matrix = haversine_km(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
bearing_matrix = bearing_deg(lats[:, None], lons[:, None], lats[None, :], lons[None, :])
np.fill_diagonal(distance_matrix, np.inf)

# ---------------- Candidate edges (fixed structure, distance-decay weighted) ----------------
within_threshold = distance_matrix <= DISTANCE_THRESHOLD_KM
edge_i, edge_j = np.where(within_threshold)

candidate_edges = pd.DataFrame({
    "Station_i": [station_order[k] for k in edge_i],
    "Station_j": [station_order[k] for k in edge_j],
    "i_idx": edge_i,
    "j_idx": edge_j,
    "distance_km": distance_matrix[edge_i, edge_j],
    "bearing_ij": bearing_matrix[edge_i, edge_j],
})
candidate_edges["distance_decay"] = np.exp(-candidate_edges["distance_km"] / DISTANCE_THRESHOLD_KM)
print(f"{len(candidate_edges)} candidate directed edges within {DISTANCE_THRESHOLD_KM:.1f} km "
      f"({len(candidate_edges) // 2} station pairs, avg out-degree {len(candidate_edges) / n_stations:.1f})")

# ---------------- Hourly wind per station (corrected "blowing toward" bearing) ----------------
wind_hourly = df[["Datetime", "Station_ID", "windspeed_10m", "winddirection_10m"]].copy()
wind_bearing_toward = (wind_hourly["winddirection_10m"] + 180) % 360
toward_rad = np.radians(wind_bearing_toward)
wind_hourly["U"] = wind_hourly["windspeed_10m"] * np.sin(toward_rad)
wind_hourly["V"] = wind_hourly["windspeed_10m"] * np.cos(toward_rad)

U_wide = wind_hourly.pivot(index="Datetime", columns="Station_ID", values="U")[station_order]
V_wide = wind_hourly.pivot(index="Datetime", columns="Station_ID", values="V")[station_order]
hourly_index = U_wide.index

wind_speed_wide = np.hypot(U_wide.to_numpy(), V_wide.to_numpy())
wind_bearing_wide = (np.degrees(np.arctan2(U_wide.to_numpy(), V_wide.to_numpy())) + 360) % 360
print(f"Hourly wind computed for {len(hourly_index):,} hours x {n_stations} stations")

# ---------------- Reference speed for the smooth speed_factor (data-derived, not guessed) ----------------
nonzero_speeds = wind_speed_wide[wind_speed_wide > 0.1]
REFERENCE_SPEED = float(np.percentile(nonzero_speeds, 75))
print(f"REFERENCE_SPEED (75th percentile wind speed): {REFERENCE_SPEED:.2f}")

# ---------------- On-demand hourly edge-weight function ----------------
def circular_diff(a, b):
    d = np.abs(a - b) % 360
    return np.minimum(d, 360 - d)

def edge_weights_at(hour_idx: int) -> pd.DataFrame:
    """Directed edge weights for a single hour (by position in hourly_index)."""
    i_idx = candidate_edges["i_idx"].to_numpy()
    wind_bearing_i = wind_bearing_wide[hour_idx, i_idx]
    wind_speed_i = wind_speed_wide[hour_idx, i_idx]

    angle_diff = circular_diff(wind_bearing_i, candidate_edges["bearing_ij"].to_numpy())
    alignment_signed = np.cos(np.radians(angle_diff))
    speed_factor = np.minimum(wind_speed_i / REFERENCE_SPEED, 1.0)  # smooth ramp, subsumes the old calm-wind gate

    weight = candidate_edges["distance_decay"].to_numpy() * (1 + WIND_WEIGHT * alignment_signed * speed_factor)

    out = candidate_edges[["Station_i", "Station_j", "bearing_ij"]].copy()
    out["Datetime"] = hourly_index[hour_idx]
    out["alignment"] = alignment_signed
    out["speed_factor"] = speed_factor
    out["weight"] = weight
    return out

# ---------------- Sanity check: build one hour's graph and inspect it ----------------
example = edge_weights_at(0)
print(f"\nExample hour ({hourly_index[0]}): {len(example)} directed edges, "
      f"weight range [{example['weight'].min():.3f}, {example['weight'].max():.3f}]")
example.head()

# ---------------- Persist the stable/reusable pieces ----------------
candidate_edges.to_pickle("/Users/drewbaldwin/PM2_5 Research/candidate_edges.pkl")
np.savez(
    "/Users/drewbaldwin/PM2_5 Research/hourly_wind_wide.npz",
    wind_speed=wind_speed_wide.astype("float32"),
    wind_bearing=wind_bearing_wide.astype("float32"),
    station_order=np.array(station_order),
    hourly_index=hourly_index.values.astype("datetime64[ns]"),
)
print("\nSaved candidate_edges.pkl and hourly_wind_wide.npz")


5874 candidate directed edges within 57.6 km (2937 station pairs, avg out-degree 33.4)
Hourly wind computed for 52,608 hours x 176 stations
REFERENCE_SPEED (75th percentile wind speed): 13.20

Example hour (2016-01-01 00:00:00): 5874 directed edges, weight range [0.327, 1.054]

Saved candidate_edges.pkl and hourly_wind_wide.npz


In [18]:
import numpy as np
import pandas as pd

IN_PATH = "/Users/drewbaldwin/PM2_5 Research/air_korea_final_imputed.pkl"
OUT_PATH = "/Users/drewbaldwin/PM2_5 Research/model_features.pkl"

df = pd.read_pickle(IN_PATH)

# ---------------- Time-varying weather features: anomaly (z-score) vs. that hour's network mean ----------------
ANOMALY_FEATURES = ["temperature_2m", "relative_humidity_2m", "surface_pressure", "windspeed_10m"]

anomaly_series = {}
hourly_mean_series = {}
for feat in ANOMALY_FEATURES:
    wide = df.pivot(index="Datetime", columns="Station_ID", values=feat)
    hourly_mean = wide.mean(axis=1)
    hourly_std = wide.std(axis=1)
    anomaly = wide.sub(hourly_mean, axis=0).div(hourly_std + 1e-6, axis=0)
    anomaly_series[f"{feat}_anomaly"] = anomaly.stack()
    hourly_mean_series[f"{feat}_hourly_mean"] = hourly_mean

weather_anomaly = pd.concat(anomaly_series, axis=1).reset_index()
weather_anomaly.columns = ["Datetime", "Station_ID"] + list(anomaly_series.keys())

# Hourly network-mean context features -- standardized here too (z-score over the full time
# series), since raw physical units (e.g. surface pressure ~1000 hPa) sit next to unit-scale
# anomaly/static features otherwise and dominate gradients during training.
hourly_means = pd.DataFrame(hourly_mean_series)
hourly_means = (hourly_means - hourly_means.mean()) / (hourly_means.std() + 1e-6)
hourly_means = hourly_means.reset_index()

# ---------------- Precipitation: handled separately ----------------
precip = df[["Datetime", "Station_ID", "precipitation"]].copy()
precip["precipitation_log1p"] = np.log1p(precip["precipitation"])
precip_mean, precip_std = precip["precipitation_log1p"].mean(), precip["precipitation_log1p"].std()
precip["precipitation_feature"] = (precip["precipitation_log1p"] - precip_mean) / precip_std
precip = precip[["Datetime", "Station_ID", "precipitation_feature"]]

# ---------------- Static features: z-score across the 176 stations (time-invariant) ----------------
STATIC_FEATURES = [
    "elevation_m", "major_roads_count_3km", "total_road_length_3km",
    "urban_landuse_area_m2_3km", "green_space_area_3km", "building_footprint_area_3km",
    "railway_length_3km", "dist_to_coast_km", "dist_to_major_road_km",
    "industrial_area_m2_3km", "traffic_points_count_3km",
]
static = df[["Station_ID"] + STATIC_FEATURES].drop_duplicates("Station_ID").reset_index(drop=True)
static_mean = static[STATIC_FEATURES].mean()
static_std = static[STATIC_FEATURES].std()
static_z = (static[STATIC_FEATURES] - static_mean) / static_std
static_z.columns = [f"{c}_z" for c in STATIC_FEATURES]
static_z = pd.concat([static[["Station_ID"]], static_z], axis=1)

static_scaler = pd.DataFrame({"mean": static_mean, "std": static_std})
static_scaler.to_pickle("/Users/drewbaldwin/PM2_5 Research/static_feature_scaler.pkl")

# ---------------- Target: PM2.5, same anomaly treatment ----------------
pm25_wide = df.pivot(index="Datetime", columns="Station_ID", values="PM25")
pm25_hourly_mean = pm25_wide.mean(axis=1)
pm25_hourly_std = pm25_wide.std(axis=1)
pm25_anomaly = pm25_wide.sub(pm25_hourly_mean, axis=0).div(pm25_hourly_std + 1e-6, axis=0)

pm25_target = pm25_anomaly.stack().rename("PM25_anomaly").reset_index()
pm25_target.columns = ["Datetime", "Station_ID", "PM25_anomaly"]

pm25_scaler = pd.DataFrame({
    "PM25_hourly_mean": pm25_hourly_mean, "PM25_hourly_std": pm25_hourly_std,
}).reset_index()
pm25_scaler.to_pickle("/Users/drewbaldwin/PM2_5 Research/pm25_anomaly_scaler.pkl")

# ---------------- Combine into one feature table ----------------
features = weather_anomaly.merge(hourly_means, on="Datetime", how="left")
features = features.merge(precip, on=["Datetime", "Station_ID"], how="left")
features = features.merge(static_z, on="Station_ID", how="left")
features = features.merge(pm25_target, on=["Datetime", "Station_ID"], how="left")

assert features.isna().sum().sum() == 0, "unexpected NaNs in prepared features"
print(f"Prepared features: {features.shape}")
print(features.columns.tolist())

features.to_pickle(OUT_PATH)
print(f"\nSaved {OUT_PATH}")
print("Also saved: static_feature_scaler.pkl, pm25_anomaly_scaler.pkl")


Prepared features: (9259008, 23)
['Datetime', 'Station_ID', 'temperature_2m_anomaly', 'relative_humidity_2m_anomaly', 'surface_pressure_anomaly', 'windspeed_10m_anomaly', 'temperature_2m_hourly_mean', 'relative_humidity_2m_hourly_mean', 'surface_pressure_hourly_mean', 'windspeed_10m_hourly_mean', 'precipitation_feature', 'elevation_m_z', 'major_roads_count_3km_z', 'total_road_length_3km_z', 'urban_landuse_area_m2_3km_z', 'green_space_area_3km_z', 'building_footprint_area_3km_z', 'railway_length_3km_z', 'dist_to_coast_km_z', 'dist_to_major_road_km_z', 'industrial_area_m2_3km_z', 'traffic_points_count_3km_z', 'PM25_anomaly']

Saved /Users/drewbaldwin/PM2_5 Research/model_features.pkl
Also saved: static_feature_scaler.pkl, pm25_anomaly_scaler.pkl


In [14]:
import numpy as np
import pandas as pd
from scipy import stats

LAG = 1
CALM_WIND_SPEED = 1.0
MIN_HOURS_PER_BIN = 500  # need enough hours in both bins for a stable per-edge correlation estimate

def circular_diff(a, b):
    d = np.abs(a - b) % 360
    return np.minimum(d, 360 - d)

results = []
for _, row in candidate_edges.iterrows():
    i, j = station_index[row["Station_i"]], station_index[row["Station_j"]]
    bearing_ij = row["bearing_ij"]

    wind_bearing_i = wind_bearing_wide[:-LAG, i]
    wind_speed_i = wind_speed_wide[:-LAG, i]
    x = resid_values[:-LAG, i]
    y = resid_values[LAG:, j]

    angle_diff = circular_diff(wind_bearing_i, bearing_ij)
    alignment_signed = np.cos(np.radians(angle_diff))
    alignment_signed = np.where(wind_speed_i < CALM_WIND_SPEED, np.nan, alignment_signed)  # drop calm hours

    valid = ~np.isnan(x) & ~np.isnan(y) & ~np.isnan(alignment_signed)
    x, y, alignment_signed = x[valid], y[valid], alignment_signed[valid]

    aligned_mask = alignment_signed > 0.5
    opposed_mask = alignment_signed < -0.5

    if aligned_mask.sum() < MIN_HOURS_PER_BIN or opposed_mask.sum() < MIN_HOURS_PER_BIN:
        continue

    aligned_corr = np.corrcoef(x[aligned_mask], y[aligned_mask])[0, 1]
    opposed_corr = np.corrcoef(x[opposed_mask], y[opposed_mask])[0, 1]

    results.append({
        "Station_i": row["Station_i"], "Station_j": row["Station_j"],
        "distance_km": row["distance_km"],
        "n_aligned": int(aligned_mask.sum()), "n_opposed": int(opposed_mask.sum()),
        "aligned_corr": aligned_corr, "opposed_corr": opposed_corr,
        "diff": aligned_corr - opposed_corr,
    })

within_pair = pd.DataFrame(results)
print(f"{len(within_pair)} directed edges with enough data in both bins (of {len(candidate_edges)} candidates)")
print(f"\nMean within-edge (aligned - opposed) correlation difference: {within_pair['diff'].mean():.4f}")
print(f"Median difference: {within_pair['diff'].median():.4f}")
print(f"% of edges where aligned > opposed: {(within_pair['diff'] > 0).mean() * 100:.1f}%")

t_stat, p_value = stats.ttest_1samp(within_pair["diff"], 0)
w_stat, w_p = stats.wilcoxon(within_pair["diff"])
print(f"\nOne-sample t-test on differences: t={t_stat:.2f}, p={p_value:.2e}")
print(f"Wilcoxon signed-rank test: p={w_p:.2e}")

within_pair.to_pickle("/Users/drewbaldwin/PM2_5 Research/within_pair_wind_test.pkl")


5874 directed edges with enough data in both bins (of 5874 candidates)

Mean within-edge (aligned - opposed) correlation difference: 0.0298
Median difference: 0.0278
% of edges where aligned > opposed: 62.9%

One-sample t-test on differences: t=28.30, p=2.95e-165
Wilcoxon signed-rank test: p=2.34e-146


In [15]:
LAG = 1
CALM_WIND_SPEED = 1.0
MIN_HOURS_PER_BIN = 500

results = []
for _, row in candidate_edges.iterrows():
    i, j = station_index[row["Station_i"]], station_index[row["Station_j"]]
    bearing_ij = row["bearing_ij"]

    wind_bearing_i = wind_bearing_wide[:-LAG, i]
    wind_speed_i = wind_speed_wide[:-LAG, i]
    x = resid_values[:-LAG, i]
    y = resid_values[LAG:, j]

    angle_diff = circular_diff(wind_bearing_i, bearing_ij)
    alignment_signed = np.cos(np.radians(angle_diff))

    valid = ~np.isnan(x) & ~np.isnan(y) & ~np.isnan(alignment_signed) & (wind_speed_i >= CALM_WIND_SPEED)
    x, y, alignment_signed, speed = x[valid], y[valid], alignment_signed[valid], wind_speed_i[valid]

    # restrict to aligned hours only -- direction is already favorable, isolate the speed effect
    aligned_mask = alignment_signed > 0.5
    if aligned_mask.sum() < 2 * MIN_HOURS_PER_BIN:
        continue
    xa, ya, speed_a = x[aligned_mask], y[aligned_mask], speed[aligned_mask]

    median_speed = np.median(speed_a)
    high_mask = speed_a >= median_speed
    low_mask = speed_a < median_speed

    if high_mask.sum() < MIN_HOURS_PER_BIN or low_mask.sum() < MIN_HOURS_PER_BIN:
        continue

    high_corr = np.corrcoef(xa[high_mask], ya[high_mask])[0, 1]
    low_corr = np.corrcoef(xa[low_mask], ya[low_mask])[0, 1]

    results.append({
        "Station_i": row["Station_i"], "Station_j": row["Station_j"],
        "distance_km": row["distance_km"],
        "n_high": int(high_mask.sum()), "n_low": int(low_mask.sum()),
        "high_speed_corr": high_corr, "low_speed_corr": low_corr,
        "diff": high_corr - low_corr,
    })

speed_test = pd.DataFrame(results)
print(f"{len(speed_test)} directed edges with enough aligned-hour data in both speed halves")
print(f"\nMean within-edge (high-speed - low-speed) correlation difference: {speed_test['diff'].mean():.4f}")
print(f"Median difference: {speed_test['diff'].median():.4f}")
print(f"% of edges where high-speed > low-speed: {(speed_test['diff'] > 0).mean() * 100:.1f}%")

from scipy import stats
t_stat, p_value = stats.ttest_1samp(speed_test["diff"], 0)
w_stat, w_p = stats.wilcoxon(speed_test["diff"])
print(f"\nOne-sample t-test: t={t_stat:.2f}, p={p_value:.2e}")
print(f"Wilcoxon signed-rank test: p={w_p:.2e}")


5874 directed edges with enough aligned-hour data in both speed halves

Mean within-edge (high-speed - low-speed) correlation difference: 0.0264
Median difference: 0.0236
% of edges where high-speed > low-speed: 66.2%

One-sample t-test: t=32.41, p=3.53e-212
Wilcoxon signed-rank test: p=2.71e-197
